# Erste Bonuslösung: Ergebnisse aller 100 Splits

CellCNN-Baseline aus 06a und **diagonale Mahalanobis-Prototypfilter mit lernbarem ReLU-Radius und Mittelwert-Pooling** aus 06b. Verwendet werden `gated_alive`, 37 Marker, `arcsinh(x/5)` und die unveränderten spenderweisen Splits 0–99 aus Aufgabe 4. Skalierung und Modellwahl verwenden ausschließlich die jeweiligen Trainingsspender. Seeds, Hyperparameter, Paketversionen und Eingabeprüfsummen stehen in `run_config.json`; die Implementierung steht in `06b_cellcnn_mahalanobis_relu_threshold.ipynb`.

Dieses Notebook liest ausschließlich die mitversionierten Tabellen unter `results/tables/bonus_100/`. Die gespeicherten Ausgaben sind ohne Ausführung sichtbar; ein frischer Kernel kann die Auswertung ohne Originaldaten, GPU oder Training wiederholen. Pro Split werden sechs Testspender gleich gewichtet. Die 600 Vorhersagen betreffen wiederholt dieselben 20 Spender und sind keine 600 unabhängigen Beobachtungen. Die Erweiterung auf 100 Splits erfolgte nach Sichtung der ersten drei Ergebnisse; der Vergleich ist deshalb deskriptiv und keine unabhängige Bestätigung.

In [1]:
from pathlib import Path
import json
import pandas as pd
from IPython.display import display

ROOT = Path.cwd()
if not (ROOT / "notebooks").is_dir():
    ROOT = ROOT.parent
TABLES = ROOT / "results/tables/bonus_100"
comparison = pd.read_csv(TABLES / "paired_comparison.csv")
predictions = pd.read_csv(TABLES / "modified_predictions.csv")
selection = pd.read_csv(TABLES / "modified_selection.csv")
config = json.loads((TABLES / "run_config.json").read_text())
assert len(comparison) == 100 and set(comparison.split_id) == set(range(100))
assert len(predictions) == 600 and predictions.groupby("split_id").size().eq(6).all()
assert not predictions.duplicated(["split_id", "donor_id"]).any()
assert len(selection) == 900 and selection.groupby("split_id").selected.sum().eq(1).all()
assert config["evaluated_split_ids"] == list(range(100))
display(pd.DataFrame({"Prüfung": ["Splits", "Testspender-Vorhersagen", "Modellkandidaten"], "Anzahl": [len(comparison), len(predictions), len(selection)]}))

Prüfung,Anzahl
Splits,100
Testspender-Vorhersagen,600
Modellkandidaten,900


## Zusammenfassung

Primär wird die Network-ROC-AUC pro Split verglichen. Die Häufigkeitsmetriken beziehen sich auf den Filter mit größtem positivem Output-Kontrast. Die Baseline nutzt das halbe Trainingsmaximum als Schwelle, die Erweiterung `d² < rho`. Fehlende positive Filter führen zu fehlenden Phänotypmetriken; sie werden nicht als null gewertet.

In [2]:
display(pd.read_csv(TABLES / "comparison_summary.csv"))
display(pd.DataFrame({"Metrik": ["Mittlere gepaarte AUC-Differenz", "Mediane gepaarte AUC-Differenz", "Splits mit höherer AUC", "Splits mit gleicher AUC", "Splits mit niedrigerer AUC"],
    "Wert": [comparison.network_auc_delta.mean(), comparison.network_auc_delta.median(),
             comparison.network_auc_delta.gt(0).sum(), comparison.network_auc_delta.eq(0).sum(),
             comparison.network_auc_delta.lt(0).sum()]}))

Metrik,Baseline,Modified
Mittlere Network-ROC-AUC,0.807500,0.636250
Mediane Network-ROC-AUC,0.875000,0.625000
Mittlere Häufigkeits-ROC-AUC,0.855867,0.587629
Mittlere Häufigkeitsdifferenz CMV+ minus CMV−,0.007403,0.026512


Metrik,Wert
Mittlere gepaarte AUC-Differenz,-0.17125
Mediane gepaarte AUC-Differenz,-0.12500
Splits mit höherer AUC,18.00000
Splits mit gleicher AUC,13.00000
Splits mit niedrigerer AUC,69.00000


## Alle 100 Splits

Positive AUC-Differenzen sprechen im jeweiligen Split für die Bonusvariante. Die überlappenden Splits werden vollständig berichtet.

In [3]:
with pd.option_context("display.max_rows", None, "display.max_columns", None):
    display(comparison[["split_id", "network_auc_baseline", "network_auc_modified", "network_auc_delta"]].rename(columns={"split_id": "Split", "network_auc_baseline": "Baseline-AUC", "network_auc_modified": "Bonus-AUC", "network_auc_delta": "AUC-Differenz"}))

Split,Baseline-AUC,Bonus-AUC,AUC-Differenz
0,1.000,0.375,-0.625
1,1.000,1.000,0.000
2,1.000,0.250,-0.750
3,1.000,0.625,-0.375
4,0.875,0.500,-0.375
5,1.000,0.500,-0.500
6,0.625,0.875,0.250
7,0.750,0.250,-0.500
8,0.500,0.375,-0.125
9,1.000,0.625,-0.375


## Phänotypassoziation

Die ergänzenden Häufigkeitsmetriken beschreiben CMV-Assoziationen und validieren keinen Zelltyp. Unterschiedliche Populationsdefinitionen schränken ihren direkten Vergleich ein.

In [4]:
display(pd.DataFrame({"Methode": ["Baseline", "Bonus"], "Bestimmbare Splits": [
    comparison.frequency_auc_baseline.notna().sum(), comparison.frequency_auc_modified.notna().sum()]}))
with pd.option_context("display.max_rows", None, "display.max_columns", None):
    display(comparison[["split_id", "frequency_auc_baseline", "frequency_auc_modified",
                        "frequency_effect_baseline", "frequency_effect_modified",
                        "phenotype_status_baseline", "phenotype_status_modified"]])

Methode,Bestimmbare Splits
Baseline,98
Bonus,97


split_id,frequency_auc_baseline,frequency_auc_modified,frequency_effect_baseline,frequency_effect_modified,phenotype_status_baseline,phenotype_status_modified
0,1.0000,0.3750,0.007350,-1.247500e-02,berechnet,berechnet
1,1.0000,0.8750,0.033825,1.150750e-01,berechnet,berechnet
2,1.0000,0.5000,0.026437,-1.040000e-02,berechnet,berechnet
3,1.0000,0.5000,0.009650,-7.427499e-02,berechnet,berechnet
4,0.8750,0.3750,0.001888,-3.964996e-02,berechnet,berechnet
5,1.0000,0.7500,0.012638,4.209998e-02,berechnet,berechnet
6,0.5000,0.7500,0.003000,7.601249e-02,berechnet,berechnet
7,0.7500,0.6250,0.000462,2.662500e-03,berechnet,berechnet
8,0.9375,NaN,0.008875,NaN,berechnet,kein positiver Output-Kontrast
9,1.0000,0.5000,0.003225,-4.275009e-03,berechnet,berechnet


## Weitere Ergebnistabellen

Die CSV-Dateien unter `results/tables/bonus_100/` enthalten zusätzlich alle 600 Spendervorhersagen, die 900 Kandidatenbewertungen, Phänotyphäufigkeiten beider Methoden, Laufzeiten und die numerischen Rekonstruktionsprüfungen der Baseline. Originaldaten und trainierte Modell-Checkpoints sind weiterhin ausschließlich lokal.